In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

bronze_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/shipments/"
)

silver_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/shipments/"
)

silver_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/orders/"
)

bronze_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/carriers/"
)

dq_invalid_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "quality/dq_invalid_records/shipments/"
)

dq_metrics_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "quality/dq_metrics/shipments/"
)

# ============================================================
# 2. LECTURE DES DONNÉES
# ============================================================

bronze_shipments = (
    spark.read
    .format("delta")
    .load(bronze_shipments_path)
)

silver_orders = (
    spark.read
    .format("delta")
    .load(silver_orders_path)
)

bronze_carriers = (
    spark.read
    .format("delta")
    .load(bronze_carriers_path)
)

print(
    "Nombre de shipments Bronze :",
    bronze_shipments.count()
)

# ============================================================
# 3. NORMALISATION
# ============================================================

normalized_shipments = (
    bronze_shipments

    .withColumn(
        "shipment_id",
        F.trim(F.col("shipment_id"))
    )

    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )

    .withColumn(
        "carrier_id",
        F.trim(F.col("carrier_id"))
    )

    .withColumn(
        "origin_city",
        F.initcap(
            F.trim(F.col("origin_city"))
        )
    )

    .withColumn(
        "destination_city",
        F.initcap(
            F.trim(F.col("destination_city"))
        )
    )

    .withColumn(
        "shipment_status",
        F.upper(
            F.trim(F.col("shipment_status"))
        )
    )
)

# ============================================================
# 4. DÉDUPLICATION
# ============================================================

dedup_window = (
    Window
    .partitionBy("shipment_id")
    .orderBy(
        F.col("updated_at").desc_nulls_last(),
        F.col("ingestion_timestamp").desc_nulls_last()
    )
)

shipments_ranked = (
    normalized_shipments
    .withColumn(
        "_row_number",
        F.row_number().over(dedup_window)
    )
)

duplicate_shipments = (
    shipments_ranked
    .filter(
        F.col("_row_number") > 1
    )
)

deduplicated_shipments = (
    shipments_ranked
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

# ============================================================
# 5. PRÉPARER LES CLÉS DE RÉFÉRENCE
# ============================================================

valid_order_ids = (
    silver_orders
    .select("order_id")
    .dropDuplicates()
    .withColumn(
        "_order_exists",
        F.lit(True)
    )
)

valid_carrier_ids = (
    bronze_carriers
    .select("carrier_id")
    .dropDuplicates()
    .withColumn(
        "_carrier_exists",
        F.lit(True)
    )
)

# ============================================================
# 6. JOINS POUR CONTRÔLER LES FOREIGN KEYS
# ============================================================

shipments_with_refs = (
    deduplicated_shipments

    .join(
        valid_order_ids,
        on="order_id",
        how="left"
    )

    .join(
        valid_carrier_ids,
        on="carrier_id",
        how="left"
    )
)

# ============================================================
# 7. RÈGLES DATA QUALITY
# ============================================================

allowed_statuses = [
    "CREATED",
    "PACKED",
    "SHIPPED",
    "HUB_SCAN",
    "OUT_FOR_DELIVERY",
    "DELIVERED"
]

validated_shipments = (
    shipments_with_refs

    .withColumn(
        "dq_shipment_id_null",
        F.col("shipment_id").isNull()
        | (F.col("shipment_id") == "")
    )

    .withColumn(
        "dq_order_id_null",
        F.col("order_id").isNull()
        | (F.col("order_id") == "")
    )

    .withColumn(
        "dq_carrier_id_null",
        F.col("carrier_id").isNull()
        | (F.col("carrier_id") == "")
    )

    .withColumn(
        "dq_order_not_found",
        F.col("_order_exists").isNull()
    )

    .withColumn(
        "dq_carrier_not_found",
        F.col("_carrier_exists").isNull()
    )

    .withColumn(
        "dq_invalid_distance",
        F.col("distance_km").isNull()
        | (F.col("distance_km") <= 0)
    )

    .withColumn(
        "dq_shipment_time_null",
        F.col("shipment_time").isNull()
    )

    .withColumn(
        "dq_expected_delivery_null",
        F.col("expected_delivery_time").isNull()
    )

    .withColumn(
        "dq_invalid_expected_time",
        (
            F.col("shipment_time").isNotNull()
            & F.col("expected_delivery_time").isNotNull()
            & (
                F.col("expected_delivery_time")
                < F.col("shipment_time")
            )
        )
    )

    .withColumn(
        "dq_invalid_actual_time",
        (
            F.col("actual_delivery_time").isNotNull()
            & F.col("shipment_time").isNotNull()
            & (
                F.col("actual_delivery_time")
                < F.col("shipment_time")
            )
        )
    )

    .withColumn(
        "dq_invalid_status",
        F.col("shipment_status").isNull()
        | (
            ~F.col("shipment_status")
            .isin(allowed_statuses)
        )
    )
)

# ============================================================
# 8. CONSTRUIRE dq_errors
# ============================================================

validated_shipments = (
    validated_shipments

    .withColumn(
        "dq_errors",
        F.array_compact(
            F.array(

                F.when(
                    F.col("dq_shipment_id_null"),
                    F.lit("SHIPMENT_ID_NULL")
                ),

                F.when(
                    F.col("dq_order_id_null"),
                    F.lit("ORDER_ID_NULL")
                ),

                F.when(
                    F.col("dq_carrier_id_null"),
                    F.lit("CARRIER_ID_NULL")
                ),

                F.when(
                    F.col("dq_order_not_found"),
                    F.lit("ORDER_NOT_FOUND")
                ),

                F.when(
                    F.col("dq_carrier_not_found"),
                    F.lit("CARRIER_NOT_FOUND")
                ),

                F.when(
                    F.col("dq_invalid_distance"),
                    F.lit("INVALID_DISTANCE")
                ),

                F.when(
                    F.col("dq_shipment_time_null"),
                    F.lit("SHIPMENT_TIME_NULL")
                ),

                F.when(
                    F.col("dq_expected_delivery_null"),
                    F.lit("EXPECTED_DELIVERY_TIME_NULL")
                ),

                F.when(
                    F.col("dq_invalid_expected_time"),
                    F.lit("EXPECTED_DELIVERY_BEFORE_SHIPMENT")
                ),

                F.when(
                    F.col("dq_invalid_actual_time"),
                    F.lit("ACTUAL_DELIVERY_BEFORE_SHIPMENT")
                ),

                F.when(
                    F.col("dq_invalid_status"),
                    F.lit("INVALID_SHIPMENT_STATUS")
                )
            )
        )
    )

    .withColumn(
        "dq_is_valid",
        F.size(F.col("dq_errors")) == 0
    )
)

# ============================================================
# 9. SÉPARER VALIDES / INVALIDES
# ============================================================

valid_shipments = (
    validated_shipments
    .filter(
        F.col("dq_is_valid") == True
    )
)

invalid_shipments = (
    validated_shipments
    .filter(
        F.col("dq_is_valid") == False
    )
)

# ============================================================
# 10. CONSTRUIRE SILVER
# ============================================================

silver_shipments = (
    valid_shipments
    .drop(
        "_order_exists",
        "_carrier_exists",
        "dq_shipment_id_null",
        "dq_order_id_null",
        "dq_carrier_id_null",
        "dq_order_not_found",
        "dq_carrier_not_found",
        "dq_invalid_distance",
        "dq_shipment_time_null",
        "dq_expected_delivery_null",
        "dq_invalid_expected_time",
        "dq_invalid_actual_time",
        "dq_invalid_status",
        "dq_errors",
        "dq_is_valid"
    )
)

# ============================================================
# 11. ÉCRITURE SILVER
# ============================================================

(
    silver_shipments
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_shipments_path)
)

# ============================================================
# 12. QUARANTAINE
# ============================================================

(
    invalid_shipments
    .write
    .format("delta")
    .mode("overwrite")
    .save(dq_invalid_shipments_path)
)

# ============================================================
# 13. MÉTRIQUES DQ
# ============================================================

records_checked = deduplicated_shipments.count()
valid_records = silver_shipments.count()
invalid_records = invalid_shipments.count()
duplicate_records = duplicate_shipments.count()

if records_checked > 0:
    dq_pass_rate = (
        valid_records
        / records_checked
    ) * 100
else:
    dq_pass_rate = 0.0

dq_metrics = [
    (
        "shipments",
        records_checked,
        valid_records,
        invalid_records,
        duplicate_records,
        float(dq_pass_rate)
    )
]

dq_metrics_schema = """
dataset STRING,
records_checked LONG,
valid_records LONG,
invalid_records LONG,
duplicate_records LONG,
dq_pass_rate DOUBLE
"""

dq_metrics_df = (
    spark.createDataFrame(
        dq_metrics,
        schema=dq_metrics_schema
    )
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
)

(
    dq_metrics_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(dq_metrics_shipments_path)
)

# ============================================================
# 14. AFFICHAGES
# ============================================================

print(
    "Bronze shipments :",
    bronze_shipments.count()
)

print(
    "Après déduplication :",
    records_checked
)

print(
    "Doublons détectés :",
    duplicate_records
)

print(
    "Shipments valides :",
    valid_records
)

print(
    "Shipments invalides :",
    invalid_records
)

print(
    "DQ pass rate :",
    round(dq_pass_rate, 2),
    "%"
)

print("\n=== SILVER SHIPMENTS ===")

display(
    silver_shipments
    .orderBy("shipment_id")
)

print("\n=== INVALID SHIPMENTS ===")

display(
    invalid_shipments.select(
        "shipment_id",
        "order_id",
        "carrier_id",
        "distance_km",
        "shipment_time",
        "expected_delivery_time",
        "actual_delivery_time",
        "shipment_status",
        "dq_errors",
        "source_file",
        "batch_id"
    )
)
display(
    invalid_shipments
    .groupBy("dq_errors")
    .count()
    .orderBy(F.desc("count"))
)

print("\n=== DQ METRICS ===")

display(
    dq_metrics_df
)
print("=== carrier_id présents dans shipments ===")

bronze_shipments.select(
    "carrier_id"
).distinct().orderBy(
    "carrier_id"
).show(truncate=False)


print("=== carrier_id présents dans carriers ===")

bronze_carriers.select(
    "carrier_id"
).distinct().orderBy(
    "carrier_id"
).show(truncate=False)


print(
    "Nombre de carriers dans Bronze :",
    bronze_carriers.count()
)





In [0]:
print("=== Nombre de violations par règle ===")

validated_shipments.select(
    F.sum(F.col("dq_shipment_id_null").cast("int"))
        .alias("shipment_id_null"),

    F.sum(F.col("dq_order_id_null").cast("int"))
        .alias("order_id_null"),

    F.sum(F.col("dq_carrier_id_null").cast("int"))
        .alias("carrier_id_null"),

    F.sum(F.col("dq_order_not_found").cast("int"))
        .alias("order_not_found"),

    F.sum(F.col("dq_carrier_not_found").cast("int"))
        .alias("carrier_not_found"),

    F.sum(F.col("dq_invalid_distance").cast("int"))
        .alias("invalid_distance"),

    F.sum(F.col("dq_shipment_time_null").cast("int"))
        .alias("shipment_time_null"),

    F.sum(F.col("dq_expected_delivery_null").cast("int"))
        .alias("expected_delivery_null"),

    F.sum(F.col("dq_invalid_expected_time").cast("int"))
        .alias("invalid_expected_time"),

    F.sum(F.col("dq_invalid_actual_time").cast("int"))
        .alias("invalid_actual_time"),

    F.sum(F.col("dq_invalid_status").cast("int"))
        .alias("invalid_status")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

display(
    bronze_shipments
    .filter(
        F.col("shipment_id") == "S0050"
    )
    .select(
        "shipment_id",
        "shipment_time",
        "expected_delivery_time",
        "distance_km",
        "shipment_status",
        "updated_at",
        "batch_id"
    )
    .orderBy("updated_at")
)